In [45]:
import json
import sys
import os
from IPython.display import Image 
from IPython.display import clear_output, display

all_images_path = "data/cub200/images/"
labeled_images_file = 'checkpoints/cub200_experiment_llama_core/classification/cub200/labeled_images.txt'

with open('checkpoints/cub200_experiment_llama_core/classification/cub200/test_samples_old.json', 'r') as f:
    test_samples = json.load(f)

with open('checkpoints/cub200_experiment_llama_core/classification/cub200/test_set200.txt', 'r') as f:
    image_files_names = f.read().splitlines()

# Load already labeled images
labeled_images = set()
if os.path.exists(labeled_images_file):
    with open(labeled_images_file, 'r') as f:
        labeled_images = set(line.strip() for line in f if line.strip())

# Filter out already labeled images
remaining_images = [img for img in image_files_names if img not in labeled_images]

scores = json.load(open('checkpoints/cub200_experiment_llama_core/classification/cub200/test_samples_evaluation.json', 'r'))
total_remaining = len(remaining_images)

if total_remaining == 0:
    print("✓ All images have been labeled!")
    sys.exit()

print(f"📊 Total images to label: {total_remaining}")
print("-" * 50)
print("\n-----------------Instructions-----------------")
print("Binary concepts for all attributes not related to markings.")
print('''1/0 for all attributes with no markings
       olive brown = gray;  orange = reddish
        with markings:
        -	Color correct: 0.7
        -	Marking correct: 0.3
        - Both correct: 1
        ''')
print("----------------------------------------------\n")


📊 Total images to label: 10
--------------------------------------------------

-----------------Instructions-----------------
Binary concepts for all attributes not related to markings.
1/0 for all attributes with no markings
       olive brown = gray;  orange = reddish
        with markings:
        -	Color correct: 0.7
        -	Marking correct: 0.3
        - Both correct: 1
        
----------------------------------------------



In [46]:
for idx, image_file in enumerate(remaining_images):
    image_folder_id = image_file.split('/')[0]
    image_file_name = image_file.split('/')[1]
    image_path = all_images_path + image_folder_id + '/' + image_file_name

    attributes = test_samples[image_file]
    image = Image(filename=image_path, width=500, height=500)
    
    for key in attributes:
        clear_output(wait=True)
        
        # Display image and current instructions
        display(image)
        print(f"\n📌 Image: {image_file}")
        print(f"🔢 Progress: {idx+1}/{total_remaining}")
        print(f"📋 Attribute: {key}")
        print(f"📊 Current value: {attributes[key]}")
        print("-" * 40)
        
        while True:
            try:
                user_input_str = input("Enter label (1=correct, 0=incorrect, or decimal for partial): ")
                if user_input_str == '':
                    print("Input cannot be empty. Please try again.")
                    continue
                user_input = float(user_input_str)
                break
            except ValueError:
                print("Invalid input. Please enter a valid number.")
        
        if user_input == -1:
            sys.exit("Evaluation stopped by user.")
        
        scores[image_file][key] = user_input

        # Save after all attributes for this image are labeled
    with open('checkpoints/cub200_experiment_llama_core/classification/cub200/test_samples_evaluation.json', 'w') as f:
        json.dump(scores, f, indent=4)
    
    # Add to labeled images file
    with open(labeled_images_file, 'a') as f:
        f.write(image_file + '\n')
    
    remaining_count = total_remaining - idx - 1
    
    clear_output(wait=True)
    print(f"✓ Image {idx+1} complete!")
    print(f"📍 Labeled: {image_file}")
    print(f"⏳ Remaining images: {remaining_count}")
    
    if remaining_count > 0:
        print("\nPress Enter to continue to the next image...")
        input()
    else:
        print("\n✓ Evaluation complete!")

print("\n✓ All images have been labeled!")

✓ Image 10 complete!
📍 Labeled: 023.Brandt_Cormorant/Brandt_Cormorant_0033_22975.jpg
⏳ Remaining images: 0

✓ Evaluation complete!

✓ All images have been labeled!
